### 1. Thiết lập môi trường và nạp dữ liệu
- Đọc `data_all.csv`.

In [ ]:
import os, sys, re, unicodedata
from collections import Counter
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

ROOT        = "/content/drive/MyDrive/project1_vsfc"
DATA_DIR    = f"{ROOT}/data"
SRC_DIR     = f"{ROOT}/src"
RESULTS_DIR = f"{ROOT}/results"

df = pd.read_csv(f"{DATA_DIR}/data_all.csv", encoding="utf-8-sig")

df.head(3)

Mounted at /content/drive


,split,text,sentiment,sentiment_name
0,dev,giáo trình chưa cụ thể .,0,negative
1,dev,giảng buồn ngủ .,0,negative
2,dev,"giáo viên vui tính , tận tâm .",2,positive


### 2. Quét dữ liệu để quyết định bảng xử lý
Ô này chỉ **đọc, không sửa dữ liệu**. Kết quả dùng để chốt 3 bảng xử lý trong `preprocess.py`:
1. **Token lạ** (emoticon, ký hiệu theo README): đếm đủ 19 token, in mỗi token một dòng để không bị cắt; kiểm tra có token nào bị dính liền vào từ khác không.
2. **Dấu thanh**: đếm các từ viết kiểu cũ (hòa, thủy) và kiểu mới (hoà, thuỷ) để thấy mức độ không nhất quán.
3. **Token `c#`**: README ghi `cshrap`, kiểm tra dữ liệu thật là `cshrap` hay `csharp`.

In [ ]:
tok = Counter(w for s in df["text"].astype(str) for w in s.split())

# 1. Token lạ theo README
special = ["colonsmile","colonsad","colonsurprise","colonlove","colonsmilesmile",
           "coloncontemn","colonbigsmile","coloncc","colonsmallsmile","coloncolon",
           "colonlovelove","colonhihi","doubledot","colonsadcolon",
           "colondoublesurprise","vdotv","dotdotdot","fraction","cshrap"]
print("=== 1. Token lạ (đứng riêng) ===")
for t in special:
    print(f"{t:22s} {tok[t]}")

# Từ nào chứa token lạ nhưng KHÔNG phải token lạ (tức là bị dính liền vào từ khác)?
stuck = {w: c for w, c in tok.items()
         if w not in special and any(t in w for t in special)}
print("\nTừ bị dính token lạ:", stuck if stuck else "không có")

# 2. Dấu thanh: kiểu cũ (òa, úy...) so với kiểu mới (oà, uý...)
TONE_PAIRS = {"òa":"oà","óa":"oá","ỏa":"oả","õa":"oã","ọa":"oạ",
              "òe":"oè","óe":"oé","ỏe":"oẻ","õe":"oẽ","ọe":"oẹ",
              "ùy":"uỳ","úy":"uý","ủy":"uỷ","ũy":"uỹ","ụy":"uỵ"}
print("\n=== 2. Dấu thanh (số lần xuất hiện của từ chứa mẫu) ===")
print(f"{'kiểu cũ':8s} {'số lần':>7s} | {'kiểu mới':8s} {'số lần':>7s}")
for old, new in TONE_PAIRS.items():
    n_old = sum(c for w, c in tok.items() if old in w)
    n_new = sum(c for w, c in tok.items() if new in w)
    if n_old or n_new:
        print(f"{old:8s} {n_old:7d} | {new:8s} {n_new:7d}")

# 3. Token c#
print("\n=== 3. Token c# ===")
for w in ["cshrap", "csharp"]:
    print(w, tok[w])

=== 1. Token lạ (đứng riêng) ===
colonsmile             39
colonsad               13
colonsurprise          2
colonlove              13
colonsmilesmile        18
coloncontemn           11
colonbigsmile          7
coloncc                1
colonsmallsmile        6
coloncolon             1
colonlovelove          1
colonhihi              7
doubledot              123
colonsadcolon          3
colondoublesurprise    4
vdotv                  12
dotdotdot              6
fraction               33
cshrap                 0

Từ bị dính token lạ: {'11doubledot55': 1, '12doubledot00': 1}

=== 2. Dấu thanh (số lần xuất hiện của từ chứa mẫu) ===
kiểu cũ   số lần | kiểu mới  số lần
òa            97 | oà           444
óa            65 | oá           110
ỏa             8 | oả           196
õa             0 | oã             1
ọa            53 | oạ           319
ỏe            13 | oẻ             0
ùy             6 | uỳ             0
úy             2 | uý            24
ủy             5 | uỷ             0
ũy 

### 3. Viết module `src/preprocess.py`
Gom toàn bộ luật tiền xử lý vào một file để dùng lại :
1. Giờ giấc: `11doubledot55` → `11:55`.
2. Dấu thanh: kiểu cũ → kiểu mới (`hòa` → `hoà`). Chiều này không làm hỏng từ hợp lệ như `toàn`, `quý`.
3. Một regex duy nhất thay viết tắt, emoticon và ký hiệu. Chuỗi dài xếp trước để `colonsmilesmile` không bị ăn nhầm bởi `colonsmile`.
4. Dọn khoảng trắng.


In [ ]:
%%writefile /content/drive/MyDrive/project1_vsfc/src/preprocess.py
"""Tiền xử lý UIT-VSFC: dùng lại được cho notebook, huấn luyện và demo."""
import re
import unicodedata

ABBR = {
    "ko": "không", "đc": "được", "trc": "trước",
    "k": "không", "khum": "không", "hông": "không", "hok": "không", "og": "không",
    "ổng": "ông ấy", "bả": "bà ấy",

    "mik": "mình", "mềnh": "mình", "mk": "mình",
    "mn": "mọi người", "mng": "mọi người",
    "g": "gì", "j": "gì", "zì": "gì",
    "b": "bạn", "bn": "bạn",
    "t": "tao", "m": "mày",

    "bit": "biết", "bít": "biết",
    "đg": "đang",
    "dc": "được", "đc": "được",
    "rùi": "rồi", "ròi": "rồi", "ùi": "rồi", "r": "rồi",
    "đượt": "được",
    "pt": "phát triển", "bt": "biết",

    # Tính từ & trạng thái cảm xúc, đánh giá
    "bth": "bình thường",
    "wa": "quá", "qá": "quá",
    "nhìu": "nhiều", "nhều": "nhiều",
    "vs": "với",
    "thik": "thích", "thít": "thích",
    "lm": "lắm",

    # Ký hiệu viết tắt tốc độ cao
    "h": "giờ",
    "đ": "đồng", "vnd": "đồng",
    "ib": "nhắn tin", "inbox": "nhắn tin",
    "add": "thêm", "rep": "phản hồi",
    "kcj": "không có gì", "kcg": "không có gì",
}

# 2. Emoticon bị README thay bằng chữ -> gom theo ý nghĩa.

EMO = {
    "colonsmile": "emosmile", "colonsmilesmile": "emosmile",
    "colonbigsmile": "emosmile", "colonsmallsmile": "emosmile",
    "colonhihi": "emosmile",
    "colonlove": "emolove", "colonlovelove": "emolove",
    "colonsad": "emosad", "colonsadcolon": "emosad",
    "colonsurprise": "emosurprise", "colondoublesurprise": "emosurprise",
    "coloncontemn": "emocontemn",
    "coloncc": "emoother",
}

# 3. Ký hiệu bị README thay bằng chữ -> trả lại ký hiệu gốc (chỉ những ký hiệu
#    VnCoreNLP KHÔNG làm vỡ). Giữ nguyên dạng chữ cái:
#      'csharp'     (README ghi 'cshrap' là sai; 'c#' bị tách thành 'c #')
#      'coloncolon' ('>>' bị tách thành '> >')
SYMBOLS = {"dotdotdot": "...", "fraction": "/", "vdotv": "v.v",
           "doubledot": ":"}

_MAP = {**ABBR, **EMO, **SYMBOLS}
# Chuỗi dài xếp trước để không ăn nhầm tiền tố (colonsmile vs colonsmilesmile)
_PATTERN = re.compile(
    r"(?<!\w)(" + "|".join(sorted(map(re.escape, _MAP), key=len, reverse=True)) + r")(?!\w)"
)
# Giờ giấc: '11doubledot55' -> '11:55' (dính liền số nên regex trên không bắt được)
_TIME = re.compile(r"(?<=\d)doubledot(?=\d)")

# 4. Dấu thanh: kiểu cũ -> kiểu mới (hòa -> hoà, thủy -> thuỷ).
#    Chiều này an toàn: âm tiết đóng (toàn, khoản, hoạt) không bao giờ viết kiểu cũ.
TONE_MAP = {
    "òa": "oà", "óa": "oá", "ỏa": "oả", "õa": "oã", "ọa": "oạ",
    "òe": "oè", "óe": "oé", "ỏe": "oẻ", "õe": "oẽ", "ọe": "oẹ",
    "ùy": "uỳ", "úy": "uý", "ủy": "uỷ", "ũy": "uỹ", "ụy": "uỵ",
}

def unify_tone(s):
    for old, new in TONE_MAP.items():
        s = s.replace(old, new)
    return s

def clean_text(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = _TIME.sub(":", s)
    s = unify_tone(s)
    s = _PATTERN.sub(lambda m: _MAP[m.group(1)], s)
    return re.sub(r"\s+", " ", s).strip()

_seg = None
def segment(s, save_dir="/content/vncorenlp"):
    """Tách từ bằng VnCoreNLP (tải model một lần duy nhất)."""
    global _seg
    if _seg is None:
        import os, py_vncorenlp
        os.makedirs(save_dir, exist_ok=True)
        if not os.path.exists(f"{save_dir}/models"):
            py_vncorenlp.download_model(save_dir=save_dir)
        _seg = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=save_dir)
    return " ".join(_seg.word_segment(s)) if s else ""

Overwriting /content/drive/MyDrive/project1_vsfc/src/preprocess.py


### 4. Áp dụng `clean_text`
-  Áp dụng lên toàn bộ 16.175 câu** để tạo cột `text_raw` (bản gốc) và `text_clean`.

In [ ]:
import importlib
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import preprocess
old_seg = getattr(preprocess, "_seg", None)   # giữ bộ tách từ nếu đã khởi động
importlib.reload(preprocess)                  # nạp lại bản mới nhất của file
preprocess._seg = old_seg
from preprocess import clean_text

df["text_raw"]   = df["text"]
df["text_clean"] = df["text_raw"].map(clean_text)

### 5. Cài VnCoreNLP
PhoBERT được huấn luyện trên văn bản đã tách từ bằng VnCoreNLP (từ ghép nối bằng `_`, ví dụ `giảng_viên`), nên cột `text_seg` phải tách bằng đúng công cụ này.
- Cài `py_vncorenlp`, kiểm tra Java. Lần gọi `segment()` đầu tiên tải model về `/content/vncorenlp`.
- Thử trên 5 câu ngẫu nhiên và một câu có từ ghép chắc chắn (`assert`).
- Kiểm tra mọi token tự đặt (`emosmile`, `emosad`, `emolove`, `emosurprise`, `emocontemn`, `emoother`, `csharp`): sau khi tách từ, mỗi token phải còn nguyên vẹn, đứng một mình.
- Thử một câu có đủ ký hiệu (`...`, `v.v`, `>>`, `11:55`, `8 / 10`, `50%`) để xem VnCoreNLP xử lý ra sao. Phần này chỉ quan sát, chưa assert.
- In thư mục làm việc trước và sau khi nạp (thư viện đổi sang `/content/vncorenlp`, nên từ giờ chỉ dùng đường dẫn tuyệt đối từ `ROOT`).

In [ ]:
!pip -q install py_vncorenlp
!java -version 2>&1 | grep -i version

print("Thư mục làm việc trước khi nạp:", os.getcwd())

# 1. Xem 5 câu ngẫu nhiên
for s in df["text_clean"].sample(5, random_state=1):
    print("clean:", s)
    print("seg  :", preprocess.segment(s))
    print()

print("Thư mục làm việc sau khi nạp:", os.getcwd())

# 2. Từ ghép chắc chắn phải được nối bằng "_"
out = preprocess.segment("giảng viên dạy rất nhiệt tình , sinh viên dễ hiểu bài .")
assert "giảng_viên" in out and "sinh_viên" in out, f"Tách từ lạ: {out}"

# 3. Mọi token tự đặt phải còn nguyên vẹn sau khi tách từ
my_tokens = sorted(set(preprocess.EMO.values())) + ["csharp"]
for t in my_tokens:
    seg_out = preprocess.segment(f"thầy dạy tốt , {t} .")
    assert t in seg_out.split(), f"VnCoreNLP làm hỏng token {t}: {seg_out}"
print("Token tự đặt giữ nguyên vẹn:", my_tokens)

# 4. Quan sát các ký hiệu (chỉ xem kết quả; assert riêng cho coloncolon)
probe = clean_text("hẹn 11doubledot55 dotdotdot vdotv coloncolon , 8 fraction 10 , đạt 50% , ví dụ doubledot A .")
print("\nclean:", probe)
print("seg  :", preprocess.segment(probe))
assert "coloncolon" in preprocess.segment(probe).split(), "coloncolon bị tách"
print("Test tách từ đạt.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
openjdk version "21.0.12.1" 2026-08-18
Thư mục làm việc trước khi nạp: /content
clean: cô vui tính , nhiệt tình .
seg  : cô vui_tính , nhiệt_tình .

clean: phòng giảng đường quá nóng , bảng ở xa nên không nhìn thấy chữ ở trên bảng .
seg  : phòng giảng_đường quá nóng , bảng ở xa nên không nhìn thấy chữ ở trên bảng .

clean: thầy hài hước .
seg  : thầy hài_hước .

clean: ví dụ , dẫn chứng cụ thể sinh động liên kết bài học .
seg  : ví_dụ , dẫn_chứng cụ_thể sinh_động liên_kết bài_học .

clean: không có ý kiến gì .
seg  : không có ý_kiến gì .

Thư mục làm việc sau khi nạp: /content/vncorenlp
Token tự đặt giữ nguyên vẹn: ['emocontemn', 'emolove', 'emoother', 'emosad', 'emosmile', 'emosurprise', 'csharp']

clean: hẹn 11:55 ... v.v coloncolon , 8 / 10 , đạt 50% , ví dụ : a .
seg  : hẹn 11:55 ... v.v coloncolon , 8 / 10 , đạt 50% , ví_dụ : a .
Test tách từ đạt.


### 6. Tách từ toàn bộ dữ liệu để tạo `text_seg`
Áp dụng `segment()` (VnCoreNLP) lên cột `text_clean`.
- **Kiểm tra cứng (`assert`)**: `text_clean` khớp bản `preprocess.py` mới; không câu nào rỗng sau khi tách; 8 token tự đặt (`emo*`, `csharp`, `coloncolon`) có số lần xuất hiện y nguyên trước và sau khi tách.
- **Kiểm tra quan sát**: bỏ `_` và khoảng trắng thì `text_seg` phải trùng `text_clean` (tách từ chỉ được thêm `_`, không đổi chữ); thống kê độ dài sau khi tách; tỉ lệ từ ghép; 5 câu mẫu.

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

# 0. text_clean phải khớp bản preprocess.py mới (phòng trường hợp quên chạy lại Ô 4)
assert (df["text_raw"].map(clean_text) == df["text_clean"]).all(), "text_clean đang cũ, hãy chạy lại Ô 4"

# 1. Tách từ
df["text_seg"] = df["text_clean"].progress_map(preprocess.segment)

# 2. Kiểm tra cứng
assert (df["text_seg"].str.len() > 0).all(), "Có câu rỗng sau khi tách từ"

tok_clean = Counter(w for s in df["text_clean"] for w in s.split())
tok_seg   = Counter(w for s in df["text_seg"]   for w in s.split())
my_tokens = sorted(set(preprocess.EMO.values())) + ["csharp", "coloncolon"]
for t in my_tokens:
    assert tok_clean[t] == tok_seg[t], f"Token {t}: trước {tok_clean[t]}, sau {tok_seg[t]}"
print("Token tự đặt giữ nguyên số lần:", {t: tok_seg[t] for t in my_tokens})

# 3. Kiểm tra quan sát: tách từ chỉ được thêm '_', không đổi chữ
strip = lambda s: s.replace(" ", "").replace("_", "")
mismatch = df[df["text_clean"].map(strip) != df["text_seg"].map(strip)]
print("Câu mà text_seg khác text_clean (sau khi bỏ '_' và khoảng trắng):", len(mismatch))
if len(mismatch):
    print(mismatch[["text_clean", "text_seg"]].head(5).to_string())

n_clean = df["text_clean"].str.split().str.len()
n_seg   = df["text_seg"].str.split().str.len()
print(pd.DataFrame({"số tiếng (text_clean)": n_clean, "số token (text_seg)": n_seg})
        .describe(percentiles=[.5, .9, .95, .99]).round(1))
ratio = sum(c for w, c in tok_seg.items() if "_" in w) / sum(tok_seg.values())
print(f"Tỉ lệ token là từ ghép (có '_'): {ratio*100:.1f}%")

print(df[["text_clean", "text_seg"]].sample(5, random_state=2).to_string())

  0%|          | 0/16175 [00:00<?, ?it/s]

Token tự đặt giữ nguyên số lần: {'emocontemn': 11, 'emolove': 14, 'emoother': 1, 'emosad': 16, 'emosmile': 77, 'emosurprise': 6, 'csharp': 5, 'coloncolon': 1}
Câu mà text_seg khác text_clean (sau khi bỏ '_' và khoảng trắng): 0
       số tiếng (text_clean)  số token (text_seg)
count                16175.0              16175.0
mean                    14.2                 11.1
std                     10.1                  8.1
min                      2.0                  2.0
50%                     11.0                  9.0
90%                     26.0                 21.0
95%                     33.0                 27.0
99%                     52.0                 42.0
max                    161.0                134.0
Tỉ lệ token là từ ghép (có '_'): 27.7%
                                                                                                                            text_clean                                                                                                    

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("vinai/phobert-base")

lens = df["text_seg"].map(lambda s: len(tok.encode(s)))   # đã tính <s> và </s>
print(lens.describe(percentiles=[.9, .95, .99]).round(1))
for L in [64, 128, 256]:
    print(f"max_length={L}: bị cắt {(lens > L).mean() * 100:.2f}% câu")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

count    16175.0
mean        13.4
std          8.4
min          4.0
50%         11.0
90%         23.0
95%         29.0
99%         45.0
max        136.0
Name: text_seg, dtype: float64
max_length=64: bị cắt 0.21% câu
max_length=128: bị cắt 0.01% câu
max_length=256: bị cắt 0.00% câu


In [ ]:
# Đo độ dài câu theo từ (s.split()) cho BiLSTM
for col in ['text_clean']:
    n = df[col].str.split().str.len()
    print(f'\n{col}:')
    print(n.describe(percentiles=[.9, .95, .99]).round(1).to_string())
    for L in [32, 48, 64, 128]:
        print(f'  max_len={L}: bị cắt {(n > L).mean() * 100:.2f}% câu')


text_clean:
count    16175.0
mean        14.2
std         10.1
min          2.0
50%         11.0
90%         26.0
95%         33.0
99%         52.0
max        161.0
  max_len=32: bị cắt 5.48% câu
  max_len=48: bị cắt 1.29% câu
  max_len=64: bị cắt 0.40% câu
  max_len=128: bị cắt 0.01% câu


### 7. Lưu `data_clean.csv`


In [ ]:
OUT  = f"{DATA_DIR}/data_clean.csv"
cols = ["split", "text_raw", "text_clean", "text_seg",
        "sentiment", "sentiment_name"]
df[cols].to_csv(OUT, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUT} ({os.path.getsize(OUT)/1e6:.1f} MB)")

Đã lưu: /content/drive/MyDrive/project1_vsfc/data/data_clean.csv (4.1 MB)
